# Chapter 03-06 · Lines, slopes, and logarithms

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** gentle - school algebra, with the
emphasis on what the numbers *mean* rather than on manipulating them

**Prerequisites:** 03-01. The probability chapters are not needed.

**Position in the learning path:** module 03, chapter 6 of 8. Before: **03-05**. After: **03-07**,
vectors and matrices, then **03-08**, gradients - after which you can read what fitting a model does.

---

## Why this matters

Nearly every model you will meet reports its findings as **coefficients**, and a coefficient is a
slope. Reading one correctly is a skill with three parts, and each part has a way of going wrong that
appears in published work regularly:

- **The units.** The same relationship is `15` or `8.33` depending on whether temperature is in
  Celsius or Fahrenheit. Neither number is more correct, and quoting one without its unit says
  nothing.
- **The intercept.** Fitting the data below gives an intercept of exactly zero, which reads as "no
  rentals at freezing" - a claim the data cannot support, because it contains no cold days at all.
- **The logarithm.** A coefficient of 0.70 on a logged outcome is routinely reported as "a 70%
  increase". It is **101%**.

This chapter is the vocabulary chapter for module 05. It is short and there is nothing difficult in
it, and skipping it is how people end up fitting models whose output they cannot describe in a
sentence.

## What you will be able to do

- Read a slope and an intercept, with units, and say what each claims
- Say when an intercept is meaningful and how to make it meaningful when it is not
- Explain what a logarithm does, and why log scales turn growth into straight lines
- Interpret the three log forms - logged outcome, logged input, both - in words
- Convert a log coefficient to a percentage correctly, and say when the shortcut breaks

## Warm-up: retrieve, do not reread

1. What is the fastest way to compute a posterior on paper?
2. Why did two positive tests mean 83% in one world and 5% in another?
3. What is a likelihood ratio?

<br>

*Answers: (1) invent a population, turn every rate into a count, fill in four cells, divide. (2) in
one world the test's errors were independent noise; in the other they were caused by a stable trait,
so the second test asked the same question of the same person. (3) `P(evidence | hypothesis)` divided
by `P(evidence | not hypothesis)` - the factor by which the evidence multiplies the odds.*

## A function is a rule

A **function** is a rule that turns each input into exactly one output. `f(12) = 180` means: give it
12, get 180.

Six days of bike rentals against the day's temperature. Six pairs, chosen so the arithmetic is exact.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# SYNTHETIC and exact: a perfectly straight relationship, so nothing is hidden by noise.
temperature = np.array([12, 15, 18, 21, 24, 27])   # degrees Celsius
rentals = np.array([180, 225, 270, 315, 360, 405])  # bikes rented that day

days = pd.DataFrame({"temperature_c": temperature, "rentals": rentals})
days["change in temp"] = days["temperature_c"].diff()
days["change in rentals"] = days["rentals"].diff()
print(days.to_string(index=False))

### Predict before running

Every three degrees, rentals rise by exactly 45.

1. What is the rise **per single degree**?
2. Write the rule that turns any temperature into a rental count.
3. What does your rule predict at 0 degrees? At 40?

In [ ]:
slope, intercept = np.polyfit(temperature, rentals, 1)
intercept = round(float(intercept), 9) + 0.0   # polyfit leaves a tiny negative residue here

print("slope     : %.4f rentals per degree Celsius" % slope)
print("intercept : %.4f rentals at 0 degrees" % intercept)
print()
print("the rule:  rentals = %.1f + %.1f x temperature" % (intercept, slope))
print()
for t in [0, 12, 20, 40]:
    print("  at %2d degrees the rule predicts %6.1f rentals" % (t, intercept + slope * t))

### The two numbers, in words

**The slope, 15.** *"Each additional degree is associated with 15 more rentals."* A slope is always a
rate: **units of y per unit of x**, here rentals per degree. Say it that way and you cannot misread
it.

**The intercept, 0.** *"At 0 degrees, 15 x 0 = 0, so the rule predicts no rentals."*

The slope is a real finding about this data. The intercept is not - and the next section is about
why.

## Failure lab: the intercept is a place the data has never been

The coldest day in the dataset is **12 degrees**. The intercept describes 0 degrees, which is four
degrees below anything observed and a different kind of weather entirely.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
line_x = np.linspace(0, 30, 100)
ax.plot(line_x, intercept + slope * line_x, color="#0072B2", linewidth=2)
ax.plot(temperature, rentals, "o", color="black", markersize=7, zorder=3)
ax.axvspan(0, 12, color="#D55E00", alpha=0.12)
ax.text(1.2, 330, "no data here\nand no reason to\nbelieve the line", color="#D55E00", fontsize=9)
ax.set_xlabel("temperature (Celsius)")
ax.set_ylabel("rentals")
ax.set_title("The fit is exact where there is data, and a guess where there is not")
ax.set_xlim(0, 30)
plt.tight_layout()
plt.show()

The line is **perfect** on the six observed days - the fit is exact, the residuals are zero, and no
diagnostic will complain. That says nothing at all about 0 degrees, where the real relationship is
almost certainly different: below freezing, rentals do not follow the summer trend down to zero, they
collapse for reasons the summer data never saw.

**An intercept is a prediction at x = 0, and it is only meaningful when x = 0 is a real, observed
situation.** For temperature, income, age or year it usually is not, and the intercept is then a
bookkeeping number that positions the line - necessary for the arithmetic, not a claim about the
world.

### The fix, which costs one line and makes the intercept useful

Subtract the mean from the input. The line does not move; the labelling does.

In [ ]:
centred = temperature - temperature.mean()
slope_c, intercept_c = np.polyfit(centred, rentals, 1)

print("original : rentals = %8.4f + %.4f x temperature" % (intercept, slope))
print("centred  : rentals = %8.4f + %.4f x (temperature - %.1f)"
      % (intercept_c, slope_c, temperature.mean()))
print()
print("the slope is unchanged: %.4f" % slope_c)
print("the intercept is now the prediction at the AVERAGE temperature")
print("  and the average rentals are %.1f" % rentals.mean())

**The intercept becomes 292.5, which is exactly the mean number of rentals**, and now it means
something: *the rentals on a typical day*. The slope is untouched at 15, because centring shifts the
line sideways without tilting it.

This is worth making a habit. **Centre your inputs and the intercept becomes interpretable for free.**
It also matters in module 05 for a second reason - it makes interaction terms and regularisation
behave sensibly - but interpretability alone is enough to justify it.

## The other thing that changes a slope without changing anything

Measure the same days in Fahrenheit.

In [ ]:
fahrenheit = temperature * 9 / 5 + 32
slope_f, intercept_f = np.polyfit(fahrenheit, rentals, 1)

print("Celsius    : rentals = %9.4f + %.4f x temp_c" % (intercept, slope))
print("Fahrenheit : rentals = %9.4f + %.4f x temp_f" % (intercept_f, slope_f))
print()
print("15 x 5/9 = %.4f   <- the same relationship, rescaled" % (15 * 5 / 9))
print()
print("prediction at 20 C  = %.1f" % (intercept + slope * 20))
print("prediction at 68 F  = %.1f   (68 F is 20 C)" % (intercept_f + slope_f * 68))

**8.3333 and 15 describe the same relationship.** Both predict 300 rentals for the same day. The
number changed because a Fahrenheit degree is smaller than a Celsius degree, so there are more of
them per rental.

**A coefficient without its units is not a number, it is a rumour.** "The effect is 8.33" is
meaningless; "8.33 rentals per Fahrenheit degree" is a finding. This becomes acute in module 05, where
a model with ten features reports ten coefficients in ten different units, and comparing their
magnitudes to decide which feature "matters most" is one of the most common mistakes in applied work -
you would be comparing rentals-per-degree with rentals-per-euro.

## Logarithms: how many times did you multiply?

A logarithm answers one question: **how many times do I multiply the base by itself to get this
number?**

`log10(1000) = 3`, because 1000 is 10 x 10 x 10. `log2(8) = 3`, because 8 is 2 x 2 x 2.

That is all. The two facts that make logarithms useful follow immediately:

- **Multiplication becomes addition.** `log(a x b) = log(a) + log(b)` - multiplying the numbers means
  adding the number of multiplications.
- **Repeated growth becomes a straight line.** Anything that grows by a fixed *percentage* per step
  is multiplying by a fixed amount per step, so its log grows by a fixed *amount* per step.

The second is why log scales exist, and it is easiest to see on something growing.

In [ ]:
# SYNTHETIC: the city's e-bike fleet, starting at 40 and growing 25% a month.
month = np.arange(0, 13)
fleet = 40 * 1.25 ** month

growth = pd.DataFrame({"month": month, "e-bikes": fleet.round(1)})
growth["multiplied by"] = (fleet / np.roll(fleet, 1)).round(3)
growth["added"] = np.concatenate([[np.nan], np.diff(fleet).round(1)])
growth.loc[0, ["multiplied by", "added"]] = np.nan
print(growth.to_string(index=False))

Look at the last two columns. The fleet **adds** a different number every month - 10, then 12.5, then
15.6 - but it **multiplies** by exactly 1.25 every month. The constant thing is the ratio, and a
logarithm is the tool that converts a constant ratio into a constant difference.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(month, fleet, "o-", color="#0072B2")
left.set_title("e-bikes: a curve")
left.set_xlabel("month")
left.set_ylabel("e-bikes")

right.plot(month, fleet, "o-", color="#0072B2")
right.set_yscale("log")
right.set_title("the same numbers on a log scale: a straight line")
right.set_xlabel("month")
right.set_ylabel("e-bikes (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
log_slope, log_intercept = np.polyfit(month, np.log(fleet), 1)

print("fitting a straight line to log(e-bikes):")
print("  slope     = %.5f    per month" % log_slope)
print("  intercept = %.5f" % log_intercept)
print()
print("exp(slope)     = %.5f   <- the monthly growth factor, and we built it as 1.25" % np.exp(log_slope))
print("exp(intercept) = %.2f      <- the starting fleet, and we built it as 40" % np.exp(log_intercept))
print()
print("doubling time = ln(2) / slope = %.2f months" % (np.log(2) / log_slope))

The log fit recovers **exactly** what we built in: a growth factor of 1.25 and a starting value of
40. A straight line in log space *is* exponential growth, and the two numbers describing the line are
the two numbers describing the growth.

**The doubling time, `ln(2) / slope`, is the most quotable form of a growth rate** - 3.11 months here.
A stakeholder who glazes over at "0.223 per month on the log scale" understands "it doubles every
three months".

## The three log forms, and what each one means

This table is the reason the chapter exists. When you fit a straight line and something has been
logged, the coefficient's *meaning* changes.

In [ ]:
forms = pd.DataFrame([
    ("y = a + b x", "neither", "one more unit of x adds b to y",
     "15 more rentals per degree"),
    ("log(y) = a + b x", "the outcome", "one more unit of x MULTIPLIES y by exp(b)",
     "each month multiplies the fleet by 1.25"),
    ("y = a + b log(x)", "the input", "each DOUBLING of x adds b x ln(2) to y",
     "each doubling of spend adds 2.08 units"),
    ("log(y) = a + b log(x)", "both", "a 1% rise in x gives a b% change in y (an elasticity)",
     "doubling x multiplies y by 2^b"),
], columns=["form", "what is logged", "how to read b", "example"])

pd.set_option("display.max_colwidth", 55)
print(forms.to_string(index=False))

In [ ]:
# y ~ log(x): each doubling adds the same amount.
spend = np.array([1, 2, 4, 8, 16, 32], dtype=float)
reach = 10 + 3 * np.log(spend)

print("y = 10 + 3 log(x)")
print(pd.DataFrame({"x (spend)": spend, "y (reach)": reach.round(4),
                    "added since last": np.concatenate([[np.nan], np.diff(reach).round(4)])})
      .to_string(index=False))
print()
print("every doubling adds exactly 3 x ln(2) = %.4f" % (3 * np.log(2)))

In [ ]:
# log(y) ~ log(x): a power law. The slope IS the exponent.
distance = np.array([10, 20, 40, 80, 160], dtype=float)
price = 500 * distance ** (-0.4)

elasticity, _ = np.polyfit(np.log(distance), np.log(price), 1)
print("y = 500 x^-0.4")
print(pd.DataFrame({"distance": distance, "price": price.round(2)}).to_string(index=False))
print()
print("slope of log(price) against log(distance) = %.4f   <- exactly the exponent" % elasticity)
print("doubling the distance multiplies the price by 2^%.1f = %.4f, a fall of %.1f%%"
      % (elasticity, 2 ** elasticity, 100 * (1 - 2 ** elasticity)))

**In a log-log fit the slope is the exponent of a power law**, and economists call it an elasticity:
a 1% change in x produces roughly a `b`% change in y. It is the one coefficient that carries no
units at all, which makes log-log fits unusually easy to compare across settings.

## Failure lab 2: a coefficient of 0.70 is not 70%

When the outcome is logged, `b` is often described as a percentage change, because for small `b`,
`exp(b) - 1` is very close to `b`. That approximation is excellent up to about 0.1 and deteriorates
fast.

**Predict before running:** a model with a logged outcome reports a coefficient of 0.70. What
percentage increase is that?

In [ ]:
rows = []
for b in [0.01, 0.05, 0.10, 0.20, 0.50, 0.70, 1.00]:
    true_percent = 100 * (np.exp(b) - 1)
    rows.append({"coefficient b": b,
                 "read naively as": "%.0f%%" % (100 * b),
                 "actually": "%.2f%%" % true_percent,
                 "error (percentage points)": round(true_percent - 100 * b, 2)})
print(pd.DataFrame(rows).to_string(index=False))

### Diagnosis

At `b = 0.05` the shortcut is off by 0.13 percentage points and nobody cares. At **`b = 0.70` it
reports 70% when the answer is 101.38%** - the effect is understated by nearly a third of itself. At
`b = 1.00` the true figure is **171.83%**.

**The rule:** convert properly with `exp(b) - 1`, always. It costs nothing and it removes an entire
category of error.

**Why the shortcut exists and why it survives:** most coefficients in practice *are* small, so the
approximation is usually harmless, and people who learned it on small coefficients apply it to large
ones without noticing the boundary. The moment a treatment effect is large - which is when it matters
most - the shortcut fails in the direction of understating your own result.

**And a second trap in the same place.** `exp(b) - 1` is right for a coefficient on a **logged
outcome**. If the outcome was logged, the model's fitted values are predictions of `log(y)`, and
`exp` of the average logged value is *not* the average of `y` - it is closer to the median. Reversing
a log transform is not free, and 02-05 met the same issue when back-transforming skewed data.

## Common misconceptions

**"The coefficient tells you how important a feature is."**
It tells you the change in y per unit of x, in whatever units x happens to be in. The same
relationship was 15 or 8.33 depending on the thermometer. Comparing raw coefficients across features
compares their units.

**"The intercept is the baseline."**
It is the prediction at x = 0, which is the baseline only if x = 0 is a real situation in your data.
Centre the inputs and it becomes the prediction at the average, which usually is what you meant.

**"A perfect fit means the model is right."**
The fit here is exact on all six days and the intercept is still a fiction. Fit quality says nothing
about the region where you have no data.

**"log means log base 10."**
In numpy, pandas, sklearn and virtually all statistical work, `log` is the **natural** logarithm, base
e. `np.log10` is the base-10 one. Getting this wrong scales every coefficient by 2.303.

**"A coefficient of 0.7 on a logged outcome is a 70% increase."**
It is 101.38%.

**"Taking logs is a trick to make data normal."**
It is a change of *what the model is additive in*. A logged outcome means the model adds effects in
percentage terms rather than absolute terms, which is a claim about how the world works - often a
good one, and always worth stating rather than doing silently.

**"You can log anything."**
Not zeros, and not negatives. Every `log1p` and every `+ 1` you have seen in someone's pipeline is a
workaround for that, and each one changes the meaning of the coefficient slightly.

## Exercises

Solutions: `solutions/03_math_foundations/03-06_functions_lines_logs_solutions.ipynb`.

### Quick understanding

**E1.** State the slope of the rentals model in a sentence that includes its units, and say what the
intercept would have to mean for it to be a real claim.

**E2.** Why does centring change the intercept but not the slope?

**E3.** In one sentence each, describe what a coefficient means in `log(y) = a + bx` and in
`y = a + b log(x)`.

### Hand calculation

**E4.** A model gives `revenue = 200 + 35 x staff`, with revenue in euros per day and staff in
people. Write the slope with units. What does the model predict for 0 staff, and is that credible?

**E5.** A fitted model on logged sales gives a coefficient of 0.35 for a promotion. Convert it to a
percentage properly. How far off is the naive reading?

**E6.** A quantity doubles every 7 years. What is its growth rate per year, as a percentage? What is
the coefficient you would see if you fitted a straight line to its natural log against years?

### Coding

**E7.** Write `describe_line(x, y, x_units, y_units)` that fits a straight line and prints the slope
and intercept as full English sentences including units, plus a warning when 0 lies outside the range
of x. Test it on the rentals data and on a case where 0 is inside the range.

**E8.** Take the e-bike fleet and add realistic noise. Fit a straight line to `fleet` against `month`,
and a straight line to `log(fleet)` against `month`. Plot both fits against the data on ordinary
axes. Which fits better, and what does the linear fit predict for month 24?

**E9.** Generate data from `y = 3 x^1.5` with noise, and recover the exponent 1.5 from a log-log fit.
Then repeat with `y = 3 x^1.5 + 20` and see what the log-log fit returns. Explain the difference.

### Interpretation

**E10.** A paper reports that a training programme raises earnings, with a coefficient of 0.62 on
logged earnings. The abstract says "a 62% increase". What should it say, and by how much has the
paper understated its own finding?

**E11.** A model predicting house prices includes both `rooms` (coefficient 15,000) and `area_m2`
(coefficient 900). A colleague says rooms matter more than area. Give two reasons that conclusion does
not follow.

### Debugging

**E12.** An analyst logs a count column with `np.log(counts)` and the pipeline produces `-inf` for
some rows and then fails. Explain the cause, give two fixes, and say what each one does to the meaning
of the coefficient.

### Exam and interview reasoning

**E13.** "When would you log-transform a variable?" Answer in five sentences, giving two situations
where you would, one where you would not, and the cost you accept when you do.

### Transfer to a different situation

**E14.** Server response time is modelled as `log(ms) = 2.1 + 0.004 x concurrent_users`. Describe in
plain words what happens as users increase, compute the response time at 100 and at 500 users, and say
at what point the service crosses one second.

### Explain it to someone non-technical

**E15.** Explain to a manager, in under 90 words, why the chart of user growth "looks like it is
slowing down" on a log scale even though the company is growing at the same rate as before.

### Optional challenge

**E16.** Fit all four forms from the chapter's table - linear, log-y, log-x, log-log - to a single
dataset generated from `y = 5 x^0.7` with multiplicative noise. Compare them on a held-out set using
an error measure computed **on the original scale of y**, and explain why comparing them by the fit
statistic in their own transformed space would be meaningless.

In [ ]:
# Your workspace. In memory: temperature, rentals, slope, intercept, centred,
# fahrenheit, month, fleet, log_slope, log_intercept, spend, reach, distance, price.

## Mastery check

- [ ] State a slope in a sentence with units, and say why the units are not optional
- [ ] Say when an intercept is a claim and when it is bookkeeping, and how to make it a claim
- [ ] Explain what a logarithm counts, in one sentence
- [ ] Recognise exponential growth from a straight line on a log scale, and extract the growth factor
- [ ] Read all four forms of the line, and convert a logged-outcome coefficient to a percentage
- [ ] Say why comparing raw coefficients across features is usually meaningless

## What should now feel instinctive

- Reading a coefficient and immediately asking "per what?"
- Checking whether x = 0 is inside the data before believing an intercept
- Centring inputs by default
- Reaching for `exp(b) - 1` rather than reading a logged coefficient as a percentage
- Seeing a curve that bends upwards and trying a log scale before anything else

## Flashcards

| Front | Back |
|---|---|
| Slope | Units of y per unit of x. Always say the units |
| Intercept | The prediction at x = 0 - a claim only if x = 0 is observed |
| Centring | Subtract the mean of x. Slope unchanged, intercept becomes the prediction at the average |
| What a logarithm counts | How many times you multiplied - so ratios become differences |
| Straight line on a log y-axis | Constant percentage growth |
| `log(y) = a + bx` | Each unit of x multiplies y by `exp(b)` |
| `y = a + b log(x)` | Each doubling of x adds `b x ln(2)` to y |
| `log(y) = a + b log(x)` | An elasticity: 1% in x gives about b% in y. The slope is the exponent |
| b = 0.70 on a logged outcome | 101.38%, not 70% |
| Doubling time | `ln(2) / growth rate` |

## Next

**03-07 · Vectors, distance, and matrices.** One feature has been enough so far. Real data has many,
and the notation for "many numbers at once" is what makes the rest of the course readable - it is
also what makes k-nearest-neighbours, clustering and every distance-based method make sense.